# 03 — Entrenamiento y evaluación

Entrena el Random Forest v1 y verifica los criterios de aceptación:
- Accuracy ≥ 80% en test set
- Inferencia < 100 ms
- Tamaño del modelo < 50 MB

In [ ]:
import time
import tempfile
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

PROCESSED_PATH = Path("../data/processed/train_data.csv")
PROCESSED_DIR = Path("../data/processed")
CLASS_NAMES = {0: "adequate", 1: "forward_slouch", 2: "excessive_recline"}

## 1. Carga de datos procesados

In [ ]:
df = pd.read_csv(PROCESSED_PATH)
X = df[["Ax1", "Ay1", "Az1"]].values
y = df["label"].values

print(f"X shape: {X.shape}")
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {CLASS_NAMES[u]}: {c} muestras")

## 2. Train / test split (80 / 20 estratificado)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")

## 3. Entrenamiento del Random Forest

In [ ]:
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
t0 = time.perf_counter()
clf.fit(X_train, y_train)
t_train = time.perf_counter() - t0
print(f"Entrenamiento completado en {t_train:.2f}s")

## 4. Evaluación

In [ ]:
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f} ({acc * 100:.2f}%)")
print()
target_names = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.title(f"Matriz de confusión (accuracy={acc:.3f})")
plt.ylabel("Real")
plt.xlabel("Predicho")
plt.tight_layout()
plt.savefig(PROCESSED_DIR / "confusion_matrix.png", dpi=100)
plt.show()

## 5. Importancia de features

In [ ]:
importances = pd.Series(clf.feature_importances_, index=["Ax1", "Ay1", "Az1"])
importances.sort_values().plot(kind="barh", color="steelblue")
plt.title("Importancia de features")
plt.xlabel("Importancia")
plt.tight_layout()
plt.savefig(PROCESSED_DIR / "feature_importance.png", dpi=100)
plt.show()
print(importances.sort_values(ascending=False))

## 6. Tiempo de inferencia

In [ ]:
sample_input = [[0.1, 9.7, 0.3]]
times = []
for _ in range(1000):
    t0 = time.perf_counter()
    clf.predict_proba(sample_input)
    times.append(time.perf_counter() - t0)

median_ms = np.median(times) * 1000
max_ms = np.max(times) * 1000
print(f"Mediana (1000 ejecuciones): {median_ms:.3f} ms")
print(f"Máximo: {max_ms:.3f} ms")

## 7. Tamaño del modelo

In [ ]:
with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as tmp:
    joblib.dump(clf, tmp.name)
    size_mb = Path(tmp.name).stat().st_size / (1024 * 1024)
    Path(tmp.name).unlink()

print(f"Tamaño estimado del modelo: {size_mb:.2f} MB")

## 8. Verificación de criterios de aceptación (ADR-004)

In [ ]:
print("=" * 45)
print("  Criterios de aceptación — ADR-004")
print("=" * 45)
ok = "✓"
fail = "✗"
print(f"  Accuracy ≥ 80%  : {ok if acc >= 0.80 else fail}  {acc*100:.2f}%")
print(f"  Inferencia <100ms: {ok if median_ms < 100 else fail}  {median_ms:.2f}ms")
print(f"  Modelo <50MB     : {ok if size_mb < 50 else fail}  {size_mb:.2f}MB")
print("=" * 45)
if acc >= 0.80 and median_ms < 100 and size_mb < 50:
    print("  ✓ Todos los criterios cumplidos. Proceder a 04-export.")
else:
    print("  ✗ Revisar los criterios que fallaron antes de exportar.")

**Siguiente paso:** `04-export.ipynb` para guardar el modelo final en `src/models/rf_v1.pkl`.